In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from source.version10.model import ResNestModel
from source.version10.score import ScoreDataset, scoreModel
from sklearn.metrics import roc_auc_score

In [3]:
def valid(fold):
    data = pd.read_csv('../../data/cdeotte/data.csv')
    data = data[data['fold'] == fold].reset_index(drop=True)
    driver = data[['image_id','source','target']].copy()
    loader = {}
    loader['path'] = '../../data/cdeotte/train/train/'
    loader['data'] = data
    valid = ScoreDataset(**loader)
    valid = DataLoader(valid, batch_size=10, shuffle=False, num_workers=6, drop_last=False)
    model = ResNestModel()
    weights = torch.load('../../model/version10/model_{}.pt'.format(fold), map_location='cpu')
    model.load_state_dict(weights['model_state_dict'])
    model = model.to('cuda:0')
    driver['score'] = scoreModel(model, valid)
    driver.to_csv('../../model/version10/valid_{}.csv'.format(fold), index=False)
    model.cpu()
    del model
    return None

In [4]:
def score(fold):
    data = pd.read_csv('../../data/cdeotte/test.csv')
    data =  data.rename(columns={'image_name':'image_id'})
    driver = data[['image_id']].copy()
    loader = {}
    loader['path'] = '../../data/cdeotte/test/test/'
    loader['data'] = data
    valid = ScoreDataset(**loader)
    valid = DataLoader(valid, batch_size=10, shuffle=False, num_workers=6, drop_last=False)
    model = ResNestModel()
    weights = torch.load('../../model/version10/model_{}.pt'.format(fold), map_location='cpu')
    model.load_state_dict(weights['model_state_dict'])
    model = model.to('cuda:0')
    driver['score'] = scoreModel(model, valid)
    driver.to_csv('../../model/version10/score_{}.csv'.format(fold), index=False)
    model.cpu()
    del model
    return None

In [5]:
valid(0)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [6]:
valid(1)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [7]:
valid(2)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [8]:
valid(3)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [9]:
valid(4)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [10]:
score(0)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [11]:
score(1)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [12]:
score(2)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [13]:
score(3)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [14]:
score(4)

Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master


In [15]:
data0 = pd.read_csv('../../model/version10/valid_0.csv')
data1 = pd.read_csv('../../model/version10/valid_1.csv')
data2 = pd.read_csv('../../model/version10/valid_2.csv')
data3 = pd.read_csv('../../model/version10/valid_3.csv')
data4 = pd.read_csv('../../model/version10/valid_4.csv')
data = data0.append(data1).append(data2).append(data3).append(data4)
data = data.groupby(['image_id','target'])['score'].mean().reset_index()
data.columns = ['image_name','target','score']
data.to_csv('../../score/version10_train_score.csv', index=False)
data.shape

(57758, 3)

In [16]:
roc_auc_score(data.target, data.score)

0.9003410743869846

In [17]:
data0 = pd.read_csv('../../model/version10/score_0.csv')
data1 = pd.read_csv('../../model/version10/score_1.csv')
data2 = pd.read_csv('../../model/version10/score_2.csv')
data3 = pd.read_csv('../../model/version10/score_3.csv')
data4 = pd.read_csv('../../model/version10/score_4.csv')
data = data0.append(data1).append(data2).append(data3).append(data4)
data = data.groupby('image_id')['score'].mean().reset_index()
data.columns = ['image_name','target']
data.to_csv('../../score/version10_test_score.csv', index=False)
data.shape

(10982, 2)

In [18]:
# 0.9425